# Automatic Differentiation with `torch.autograd`
Quando si allena una rete neurale l'algoritmo più utilizzando è *back propagation* dove i parametri vengono aggiustati in base al **gradient** della loss function.

Il ciclo, in termini semplici, è composto da:
- forward: La rete tenta di indovinare
- Loss: Calcola di quanto ha sbagliato
- Backpropagation + Gradiente: La rete calcola all'indietro come modificare i pesi per ridurre l'errore
- La rete modifica i pesi

Per calcolare i gradients, PyTorch ha un engine al suo interno chiamato `torch.autograd`.

Consideriamo la rete più semplice con un solo layer, con input `x`, parametri `w` e `b` e una loss function. Possiamo definirla in PyTorch con:

In [2]:
import torch

x = torch.ones(5) # input tensor
y = torch.zeros(3) # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

Quando eseguiamo operazioni in PyTorch, Autograd costruisce una mappa chiamata **Computational Graph** che è DAG (Directed Acyclic Graph). Per fare questo abbiamo bisogno che i tenso che contengono i pesi ovvero `b,w` vengano "tracciati" da PyTorch, per farlo impostiamo il campo `requires_grad=True`.
Ogni volta che viene eseguita un'operazione su un tensor tracciato viene creato un oggetto `Function`, questo oggetto sa due cose:
- Come eseguire il calcolo in avanti (Forward)
- Come calcolare la derivata tornando indietro (Backward).

I tensor hanno l'attributo `grad_fn` che corrisponde alla traccia delle operazioni effettuate, i tensor che sono il risultato di un'operazione avranno un valore in questo attributo che permette ad Autograd di ritrovare la strada a ritroso durante la backpropagation.

In [9]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x7f7ae7c1ceb0>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7f7ae7c1ceb0>


## Computing Gradients
Dopo aver fatto passare i dati nella rete e aver calcolato la Loss dobbiamo avviare il calcolo delle derivate.
- `loss.backward()` con questo comando Autograd parte dalla Loss e percorre il grafo al contrario basandosi sui `grad_fn` e applicando le regole delle derivate.
- `.grad`: Autograd svolgendo i calcoli per calcolare i gradienti deposita i risultati in questo attributo del tensore

In [10]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0101, 0.0614, 0.0722],
        [0.0101, 0.0614, 0.0722],
        [0.0101, 0.0614, 0.0722],
        [0.0101, 0.0614, 0.0722],
        [0.0101, 0.0614, 0.0722]])
tensor([0.0101, 0.0614, 0.0722])


Da notare che è possibile chiamare `backward` una sola volta su una mappa dato che poi questa verrà distrutta per liberare memoria, per mantenerla dobbiamo specificare `retain_graph=True`

## Disabilitare Autograd
Quando non siamo più in fase di addestramento non vogliamo che PyTorch registri le operazioni svolte sui tensor.

Per farlo possiamo inserire il codice in un blocco `with torch.no_grad()` oppure utilizzare il metodo `.detach()` sul singolo tensore.

In [11]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

True
False


In [12]:
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

False


In generale, autograd tiene dei record dei tensors su tutte le operazioni eseguite su di essi in un DAG composto da oggetti di tipo Function.
In questo DAG le foglie sono i tensors di input mentre la radice è quello di output, tracciando questo DAG dalla radice alle foglie possiamo calcolare i gradients.

In uno step di forward, autograd fa due cose simultaneamente:
- Esegue l'operazione e calcola il tensor richiesto
- Mantiene il gradient della funzione nel DAG

Quando chiamiamo `.backward()` sulla radice del DAG Autograd compie diverse azioni:
- Calcola i gradients da ogni `.grad_fn`
- Gli accumula nel corrispondente attributo `.grad`
- Si propaga fino alle foglie dell'albero.

Da notare che a differenza di altri framework, Autograd usa un grafo dinamico, la mappa quindi non è una struttura fissa decisa all'inizio, il grafo viene sempre creato da 0 mentre il codice viene eseguito. Questo ci permette con ad esempio degli if di cambiarne la forma, la dimensione o altro ad ogni iterazione.